In [60]:
from pathlib import Path
from enum import Enum

from __future__ import annotations

import json
from textwrap import dedent
from typing import TypedDict, Literal, Optional, Any, Callable, Generic, TypeVar, Iterable
import re
from pydantic import BaseModel, Field, model_validator
from uuid import uuid4

from dataclasses import dataclass, field
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [40]:
load_dotenv()

True

### Checks if the `./story/time.md` or `./story/elements.md` is not present, then creates them based on the template

In [13]:
EVENTS_MD_TEMPLATE = """# Events

Purpose:
This file is the canonical index of meaningful story events.

Definitions:
- Event:
  A bounded, world-relevant unit of change. An event is something that happens, is discovered, is decided, or is revealed in a way that meaningfully changes the story world, a character's knowledge, a relationship, a goal, a risk, or the consequences that follow. It is not merely a scene fragment or a sentence-level action. It should be stable enough to remain the same event even if later chapters refine its timing, participants, cause, or full meaning.

- UUID:
  A stable canonical identifier for the event. Its purpose is identity, not meaning. It allows the system to refer to the same event reliably across the index, the detailed event file, later updates, and agent workflows, even if the event's wording, timing, or interpretation evolves. UUIDs are generated programmatically and treated as the permanent handle for each event.

- When:
  The best currently known placement of the event in story time. Its purpose is to situate the event chronologically without forcing false precision. It may be exact, approximate, relative, inferred, or unknown, depending on what the manuscript currently supports. This field should remain open to later refinement as future chapters reveal stronger temporal grounding. It captures story-time placement, not document-edit time.

- Chapters:
  The manuscript locations where the event is evidenced, introduced, developed, or clarified. Its purpose is grounding and traceability. It tells the system and the reader where this event comes from in the text. It is an internal reference to the relevant chapter or chunk, not part of the event's meaning itself. Multiple chapters may be listed when an event is introduced in one place and clarified or extended later.

- Summary:
  A short canonical description of what the event is. Its purpose is to make the event legible at index level without retelling the full scene. It should describe the core happening in clear, neutral, world-level language, focusing on what changed or what became known. It should not drift into interpretation, prose style, emotional commentary, or speculation beyond what the manuscript supports.

Format:
- uuid | when | chapters | summary

Notes:
- one line per meaningful story event
- uuid is generated programmatically
- when may remain open if the manuscript does not yet support a stronger placement
- chapters may be a single chapter, multiple chapters, or a range
- summary should stay compact and canonical
- detailed event files live in ./story/events/<uuid>.md

## Entries
"""

In [17]:
ELEMENTS_MD_TEMPLATE = """# Elements

Purpose:
This file is the canonical index of stable, story-relevant elements in the world model.

Definitions:
- Element:
  A stable, story-relevant unit of the world model. An element is any distinct thing the story can meaningfully refer to, track, or reason about across time: a person, place, item, group, concept, rule, institution, creature, or other persistent entity. An element is not just any word that appears in the prose. It should be something with enough identity and continuity that the system benefits from treating it as a canonical object in the world.

- Kind:
  The broad category of what the element is. Its purpose is to place the element into the right conceptual bucket so the system can reason about it appropriately. Kind should be chosen at the level that is useful and stable, such as person, place, item, animal, concept, group, or other. It is a typing aid, not a full explanation of the element.

- Display Name:
  The canonical surface name used to represent the element in the index. Its purpose is readability and consistency. It is the main human-facing name by which the element is referred to in the world model. It should be the clearest stable name for the element, even if the manuscript sometimes refers to it in other ways.

- UUID:
  A stable canonical identifier for the element. Its purpose is identity, not meaning. It allows the system to refer to the same element reliably across the index, the detailed element file, future updates, and agent workflows, even if the display name, aliases, or understanding of the element evolves. UUIDs are generated programmatically and treated as the permanent handle for that element.

- Aliases:
  Alternative names, labels, titles, spellings, or surface references that may refer to the same element. Their purpose is recognition and matching. They help the system understand that multiple phrasings in the manuscript may point to one canonical element. Aliases should capture meaningful alternate references, not every descriptive phrase the prose happens to use.

- Identification Keys:
  Short recognition cues that help distinguish this element from others. Their purpose is practical identification, especially when names are ambiguous, absent, evolving, or reused. These should be compact, concrete signals grounded in the manuscript, such as roles, relationships, defining objects, repeated traits, or highly specific associations. They are not meant to be full descriptions or interpretations. They are quick anchors that help the system recognize who or what this element is.

Format:
- kind | display_name | uuid | aliases | identification_keys

Notes:
- one line per canonical element
- uuid is generated programmatically
- aliases are comma-separated
- identification_keys are semicolon-separated short recognition cues
- detailed element files live in ./story/elements/<uuid>.md

## Entries
"""

In [19]:
files_to_create = {
    Path("./story/events.md"): EVENTS_MD_TEMPLATE,
    Path("./story/elements.md"): ELEMENTS_MD_TEMPLATE,
}

In [20]:
for file_path, file_content in files_to_create.items():
    file_path.parent.mkdir(parents=True, exist_ok=True)

    if file_path.exists():
        print(f"Already present: {file_path}")
    else:
        file_path.write_text(file_content, encoding="utf-8")
        print(f"Created: {file_path}")

Already present: story/events.md
Created: story/elements.md


### Checks whether the indexed elements in `./story/elements.md` are present in the `./story/elements/<UUID>.md` and validates its content.

In [21]:
ELEMENTS_INDEX_PATH = Path("./story/elements.md")
ELEMENTS_DIRECTORY_PATH = Path("./story/elements/")

In [22]:
class ElementRecord(TypedDict):
    kind: str
    display_name: str
    uuid: str
    aliases: str
    identification_keys: str

In [23]:
ELEMENT_FILE_TEMPLATE = """# {display_name}

## Identification
- UUID: {uuid}
- Type: {kind}
- Canonical name: {display_name}
- Aliases: {aliases}
- Identification keys: {identification_keys}

## Core Understanding
TBD

## Stable Profile
- TBD

## Interpretation
- TBD

## Knowledge / Beliefs / Uncertainties
- TBD

## Element-Centered Chronology
### TBD
- TBD

## Open Threads
- TBD
"""

In [24]:
def normalize_optional_text(value: str) -> str:
    return value.strip() if value.strip() else "-"


def build_element_file_content(element_record: ElementRecord) -> str:
    template_values = {
        "kind": element_record["kind"].strip(),
        "display_name": element_record["display_name"].strip(),
        "uuid": element_record["uuid"].strip(),
        "aliases": normalize_optional_text(element_record["aliases"]),
        "identification_keys": normalize_optional_text(
            element_record["identification_keys"]
        ),
    }

    return ELEMENT_FILE_TEMPLATE.format(**template_values)

In [25]:
def parse_element_entries(elements_index_text: str) -> list[dict[str, str]]:
    element_records: list[dict[str, str]] = []
    in_entries_section = False

    for line in elements_index_text.splitlines():
        stripped_line = line.strip()

        if stripped_line == "## Entries":
            in_entries_section = True
            continue

        if not in_entries_section or not stripped_line.startswith("- "):
            continue

        parts = [part.strip() for part in stripped_line[2:].split("|")]
        if len(parts) != 5:
            continue

        kind, display_name, uuid, aliases, identification_keys = parts

        element_records.append(
            {
                "kind": kind,
                "display_name": display_name,
                "uuid": uuid,
                "aliases": aliases,
                "identification_keys": identification_keys,
            }
        )

    return element_records

In [26]:
def extract_identification_fields(element_file_text: str) -> dict[str, str]:
    field_patterns = {
        "uuid": r"^- UUID:\s*(.+)$",
        "kind": r"^- Type:\s*(.+)$",
        "display_name": r"^- Canonical name:\s*(.+)$",
        "aliases": r"^- Aliases:\s*(.+)$",
    }

    extracted_fields: dict[str, str] = {}

    for field_name, pattern in field_patterns.items():
        match = re.search(pattern, element_file_text, flags=re.MULTILINE)
        extracted_fields[field_name] = match.group(1).strip() if match else ""

    return extracted_fields

def get_expected_identification_fields(element_record: dict[str, str]) -> dict[str, str]:
    return {
        "uuid": element_record["uuid"].strip(),
        "kind": element_record["kind"].strip(),
        "display_name": element_record["display_name"].strip(),
        "aliases": normalize_optional_text(element_record["aliases"]),
    }

In [28]:
mismatch_reports: list[Path] = []

if not ELEMENTS_INDEX_PATH.exists():
    print(f"Not present: {ELEMENTS_INDEX_PATH}")
else:
    print(f"Present: {ELEMENTS_INDEX_PATH}")

    elements_index_text = ELEMENTS_INDEX_PATH.read_text(encoding="utf-8")
    element_records = parse_element_entries(elements_index_text)

    ELEMENTS_DIRECTORY_PATH.mkdir(parents=True, exist_ok=True)

    if not element_records:
        print("No valid element entries were found in elements.md.")
    else:
        for element_record in element_records:
            element_file_path = ELEMENTS_DIRECTORY_PATH / f'{element_record["uuid"]}.md'

            if not element_file_path.exists():
                element_file_content = build_element_file_content(element_record)
                element_file_path.write_text(element_file_content, encoding="utf-8")
                print(f"Created: {element_file_path}")
                continue

            current_fields = extract_identification_fields(
                element_file_path.read_text(encoding="utf-8")
            )
            expected_fields = get_expected_identification_fields(element_record)

            if current_fields == expected_fields:
                print(f"Verified: {element_file_path}")
            else:
                mismatch_reports.append(
                    {
                        "path": element_file_path,
                        "expected": expected_fields,
                        "current": current_fields,
                    }
                )
                print(f"Mismatch: {element_file_path}")

print("\nFiles needing review:")
for mismatch_report in mismatch_reports:
    print(mismatch_report["path"])

Present: story/elements.md
Verified: story/elements/elt_6ad4c9d565e1.md
Verified: story/elements/elt_f91e46621aa0.md
Verified: story/elements/elt_0302285ec15e.md
Verified: story/elements/elt_5fd5b7cea1ae.md
Verified: story/elements/elt_c24a27ea02ba.md
Verified: story/elements/elt_9761d43a7212.md
Verified: story/elements/elt_0b701b62514a.md
Verified: story/elements/elt_50b1a9dd537d.md
Verified: story/elements/elt_94a3d3a6d9b7.md
Verified: story/elements/elt_10458a6f97c0.md
Verified: story/elements/elt_f5fb610a5b2a.md
Verified: story/elements/elt_d645b4d57830.md
Verified: story/elements/elt_1d1c7c76756e.md
Verified: story/elements/elt_5776327dc313.md
Verified: story/elements/elt_333cc7b06453.md
Verified: story/elements/elt_fa9b9d0b1904.md
Verified: story/elements/elt_d5ea7dc8751d.md
Verified: story/elements/elt_37dcb83f2663.md
Verified: story/elements/elt_251c313f164a.md
Verified: story/elements/elt_f1b7cd5fa6e9.md
Verified: story/elements/elt_bf43c0cf189f.md
Verified: story/elements/elt

### Checks whether the indexed elements in `./story/events.md` are present in the `./story/events/<UUID>.md` and validates its content.

In [61]:
EVENTS_INDEX_PATH = Path("./story/events.md")
EVENTS_DIRECTORY_PATH = Path("./story/events/")

In [62]:
class EventRecord(TypedDict):
    uuid: str
    when: str
    chapters: str
    summary: str


In [63]:
EVENT_FILE_TEMPLATE = """# Event Record

## Identity
- UUID: {uuid}
- When: {when}
- Chapters: {chapter}
- Summary: {summary}

## Purpose
This file is the grounded canonical record for one bounded story event.

It exists to capture what the manuscript currently supports about this event:
- what the event is
- where it belongs in story time
- where it appears in the manuscript
- what changed because of it
- what remains uncertain

This file must stay faithful to the manuscript and the current world model.
It must not invent hidden meaning, motives, symbolism, or consequences not supported by the text.

## Definitions
- Event:
  A bounded, world-relevant unit of change. It is something that happens, is discovered, is decided, or is revealed in a way that meaningfully changes the story world, a character’s knowledge, a relationship, a goal, a risk, or the consequences that follow.

- Canonical title:
  A short neutral label that helps identify the event without retelling the scene.

- Story-time placement:
  The best currently supported placement of the event in the chronology of the story. It may remain open if precision is not supported.

- Manuscript grounding:
  The chapter references and evidence snippets that justify this event record.

- Consequence:
  A change that follows from the event and is supported by the manuscript or current canon.

- Unknown:
  Something that has not yet been established clearly enough to state as canon.


## Manuscript Grounding

## What Happened
- TBD

## What Became Known or Decided
- TBD

## Participants / Involved Elements
- Characters:
  - TBD
- Places:
  - TBD
- Objects:
  - TBD
- Other elements:
  - TBD

## Preconditions / Context
- TBD

## Consequences
- Immediate:
  - TBD
- Ongoing / downstream:
  - TBD

## Affected World-Model Areas
- TBD

## Related Events
- Previous / prerequisite events:
  - TBD
- Follow-on / resulting events:
  - TBD

## Boundaries / Unknowns
- TBD

## Revision Notes
- Created from: events.md index entry
- Last updated because: TBD
"""


In [64]:
def normalize_optional_text(value: str) -> str:
    value = value.strip()
    return value if value else "-"


def normalize_chapters_text(value: str) -> str:
    value = value.strip()
    return value if value else "-"


def chapters_text_to_block(chapters_text: str) -> str:
    chapters_text = normalize_chapters_text(chapters_text)

    if chapters_text == "-":
        return "  - -"

    parts = [part.strip() for part in chapters_text.split(",") if part.strip()]
    if not parts:
        return "  - -"

    return "\n".join(f"  - {part}" for part in parts)


In [65]:
def chapters_block_to_text(chapters_block: str) -> str:
    chapter_lines = []

    for line in chapters_block.splitlines():
        stripped = line.strip()
        if stripped.startswith("- "):
            value = stripped[2:].strip()
            if value and value != "-":
                chapter_lines.append(value)

    return ", ".join(chapter_lines) if chapter_lines else "-"

In [67]:
def build_canonical_title(summary: str) -> str:
    summary = summary.strip()
    if not summary:
        return "Untitled event"
    return summary[:80]

def build_event_file_content(event_record: EventRecord) -> str:
    template_values = {
        "uuid": event_record["uuid"].strip(),
        "canonical_title": build_canonical_title(event_record["summary"]),
        "when": normalize_optional_text(event_record["when"]),
        "summary": normalize_optional_text(event_record["summary"]),
        "chapters_block": chapters_text_to_block(event_record["chapters"]),
    }

    return EVENT_FILE_TEMPLATE.format(**template_values)

In [68]:
def parse_event_entries(events_index_text: str) -> list[EventRecord]:
    event_records: list[EventRecord] = []
    in_entries_section = False

    for line in events_index_text.splitlines():
        stripped_line = line.strip()

        if stripped_line == "## Entries":
            in_entries_section = True
            continue

        if not in_entries_section or not stripped_line.startswith("- "):
            continue

        parts = [part.strip() for part in stripped_line[2:].split("|")]
        if len(parts) != 4:
            continue

        uuid, when, chapters, summary = parts

        event_records.append(
            {
                "uuid": uuid,
                "when": when,
                "chapters": chapters,
                "summary": summary,
            }
        )

    return event_records

In [69]:
def extract_event_indexed_fields(event_file_text: str) -> dict[str, str]:
    uuid_match = re.search(r"^- UUID:\s*(.+)$", event_file_text, flags=re.MULTILINE)
    when_match = re.search(r"^- When:\s*(.+)$", event_file_text, flags=re.MULTILINE)

    summary_match = re.search(
        r"## Canonical Summary\s*\n(.+?)(?:\n## |\Z)",
        event_file_text,
        flags=re.DOTALL,
    )

    chapters_match = re.search(
        r"## Manuscript Grounding\s*\n- Chapters:\s*\n(.+?)(?:\n- Evidence:|\n## |\Z)",
        event_file_text,
        flags=re.DOTALL,
    )

    uuid = uuid_match.group(1).strip() if uuid_match else ""
    when = when_match.group(1).strip() if when_match else ""
    summary = summary_match.group(1).strip() if summary_match else ""
    chapters = chapters_block_to_text(chapters_match.group(1)) if chapters_match else ""

    return {
        "uuid": uuid,
        "when": when,
        "chapters": chapters,
        "summary": summary,
    }


def get_expected_event_indexed_fields(event_record: EventRecord) -> dict[str, str]:
    return {
        "uuid": event_record["uuid"].strip(),
        "when": normalize_optional_text(event_record["when"]),
        "chapters": normalize_chapters_text(event_record["chapters"]),
        "summary": normalize_optional_text(event_record["summary"]),
    }


In [70]:
mismatch_reports: list[dict] = []

if not EVENTS_INDEX_PATH.exists():
    print(f"Not present: {EVENTS_INDEX_PATH}")
else:
    print(f"Present: {EVENTS_INDEX_PATH}")

    events_index_text = EVENTS_INDEX_PATH.read_text(encoding="utf-8")
    event_records = parse_event_entries(events_index_text)

    EVENTS_DIRECTORY_PATH.mkdir(parents=True, exist_ok=True)

    if not event_records:
        print("No valid event entries were found in events.md.")
    else:
        for event_record in event_records:
            event_file_path = EVENTS_DIRECTORY_PATH / f'{event_record["uuid"]}.md'

            if not event_file_path.exists():
                event_file_content = build_event_file_content(event_record)
                event_file_path.write_text(event_file_content, encoding="utf-8")
                print(f"Created: {event_file_path}")
                continue

            current_fields = extract_event_indexed_fields(
                event_file_path.read_text(encoding="utf-8")
            )
            expected_fields = get_expected_event_indexed_fields(event_record)

            if current_fields == expected_fields:
                print(f"Verified: {event_file_path}")
            else:
                mismatch_reports.append(
                    {
                        "path": event_file_path,
                        "expected": expected_fields,
                        "current": current_fields,
                    }
                )
                print(f"Mismatch: {event_file_path}")

print("\nFiles needing review:")
for mismatch_report in mismatch_reports:
    print(mismatch_report["path"])

Present: story/events.md
No valid event entries were found in events.md.

Files needing review:


## Timeline Agent

In [56]:
# ============================================================
# Canonical LLM output
# ============================================================

class EventDeltaAction(str, Enum):
    CREATE = "create"
    UPDATE = "update"
    DELETE = "delete"


class CanonicalEventRecord(BaseModel):
    """
    The normalized canonical representation of an event as it should
    exist in events.md after approval.
    """

    when: str = Field(
        description=(
            "Best currently supported placement of the event in story time. "
            "May be exact, approximate, relative, inferred, or unknown."
        )
    )
    chapters: list[str] = Field(
        default_factory=list,
        description=(
            "Chapter or manuscript chunk references where this event is introduced, "
            "evidenced, developed, or clarified."
        )
    )
    summary: str = Field(
        description=(
            "Short canonical description of the event in neutral world-model language."
        )
    )


class CanonicalEventDelta(BaseModel):
    """
    A single canon-level decision produced by the model.

    This is the core unit of output:
    - create a new event
    - update an existing event
    - delete an existing event
    """

    action: EventDeltaAction = Field(
        description=(
            "Canon-level operation implied by the manuscript diff."
        )
    )

    existing_event_uuid: Optional[str] = Field(
        default=None,
        description=(
            "UUID of the matched existing canonical event for update/delete. "
            "Must be null for create."
        )
    )

    proposed_record: Optional[CanonicalEventRecord] = Field(
        default=None,
        description=(
            "The canonical record that should exist after approval. "
            "Required for create/update. Must be null for delete."
        )
    )

    reason: str = Field(
        description=(
            "Compact grounded explanation of why this delta is a create, update, "
            "or delete, including identity reasoning when applicable."
        )
    )

    evidence_from_diff: list[str] = Field(
        default_factory=list,
        description=(
            "Short exact phrases or close excerpts from the diff that justify "
            "the decision."
        )
    )

    @model_validator(mode="after")
    def validate_shape(self) -> "CanonicalEventDelta":
        if self.action == EventDeltaAction.CREATE:
            if self.existing_event_uuid is not None:
                raise ValueError(
                    "existing_event_uuid must be null when action='create'."
                )
            if self.proposed_record is None:
                raise ValueError(
                    "proposed_record is required when action='create'."
                )

        elif self.action == EventDeltaAction.UPDATE:
            if not self.existing_event_uuid:
                raise ValueError(
                    "existing_event_uuid is required when action='update'."
                )
            if self.proposed_record is None:
                raise ValueError(
                    "proposed_record is required when action='update'."
                )

        elif self.action == EventDeltaAction.DELETE:
            if not self.existing_event_uuid:
                raise ValueError(
                    "existing_event_uuid is required when action='delete'."
                )
            if self.proposed_record is not None:
                raise ValueError(
                    "proposed_record must be null when action='delete'."
                )

        return self


class EventsCanonDeltaProposal(BaseModel):
    """
    The single structured output the LLM returns.

    This represents the full set of canon-level event deltas implied by the diff.
    """

    scan_summary: str = Field(
        description=(
            "Compact summary of what the diff changed at the event-index level."
        )
    )
    deltas: list[CanonicalEventDelta] = Field(
        default_factory=list,
        description=(
            "All canon-level create/update/delete decisions implied by the diff."
        )
    )

In [55]:
# ============================================================
# Derived review models
# ============================================================

class ReviewEventRecord(BaseModel):
    """
    Reviewer-facing event record shape.
    This mirrors the canonical event record but is named separately
    to make the review layer explicit.
    """

    when: str
    chapters: list[str] = Field(default_factory=list)
    summary: str


class EventReviewDecision(BaseModel):
    """
    A reviewer-facing view of one canon delta.

    Derived from:
    - one CanonicalEventDelta
    - zero or one existing parsed event record from events.md
    """

    action: EventDeltaAction = Field(
        description="Reviewer-facing action label."
    )

    target_event_uuid: Optional[str] = Field(
        default=None,
        description=(
            "UUID of the existing event being updated/deleted. Null for creates."
        )
    )

    current_record: Optional[ReviewEventRecord] = Field(
        default=None,
        description=(
            "Current record from events.md before approval. "
            "Null for creates."
        )
    )

    proposed_record: Optional[ReviewEventRecord] = Field(
        default=None,
        description=(
            "Record that would exist after approval. "
            "Null for deletes."
        )
    )

    identity_rationale: str = Field(
        description=(
            "Why this was matched as an update/delete target or treated as new."
        )
    )

    evidence_from_diff: list[str] = Field(
        default_factory=list,
        description="Grounding snippets from the diff."
    )

    @model_validator(mode="after")
    def validate_shape(self) -> "EventReviewDecision":
        if self.action == EventDeltaAction.CREATE:
            if self.target_event_uuid is not None:
                raise ValueError(
                    "target_event_uuid must be null for create."
                )
            if self.current_record is not None:
                raise ValueError(
                    "current_record must be null for create."
                )
            if self.proposed_record is None:
                raise ValueError(
                    "proposed_record is required for create."
                )

        elif self.action == EventDeltaAction.UPDATE:
            if not self.target_event_uuid:
                raise ValueError(
                    "target_event_uuid is required for update."
                )
            if self.current_record is None:
                raise ValueError(
                    "current_record is required for update."
                )
            if self.proposed_record is None:
                raise ValueError(
                    "proposed_record is required for update."
                )

        elif self.action == EventDeltaAction.DELETE:
            if not self.target_event_uuid:
                raise ValueError(
                    "target_event_uuid is required for delete."
                )
            if self.current_record is None:
                raise ValueError(
                    "current_record is required for delete."
                )
            if self.proposed_record is not None:
                raise ValueError(
                    "proposed_record must be null for delete."
                )

        return self


class EventsReviewProposal(BaseModel):
    """
    Full reviewer-facing proposal.

    Derived after enriching the model output with current events.md records.
    """

    changed: bool = Field(
        description=(
            "Whether the diff implies any meaningful change to the canonical events index."
        )
    )

    scan_summary: str = Field(
        description="Reviewer-facing summary of the event-index impact."
    )

    review_decisions: list[EventReviewDecision] = Field(
        default_factory=list,
        description="All reviewer-facing decisions."
    )

    approval_message: str = Field(
        description=(
            "Short summary of what will happen if the reviewer approves."
        )
    )

    @model_validator(mode="after")
    def validate_changed_consistency(self) -> "EventsReviewProposal":
        if not self.changed and self.review_decisions:
            raise ValueError(
                "review_decisions must be empty when changed=False."
            )
        if self.changed and not self.review_decisions:
            raise ValueError(
                "review_decisions must not be empty when changed=True."
            )
        return self

In [57]:
# ============================================================
# Existing parsed event shape from events.md
# ============================================================

class IndexedEventRecord(BaseModel):
    """
    Deterministic parsed representation of one current event entry
    from events.md, keyed by UUID outside this model.
    """

    when: str
    chapters: list[str] = Field(default_factory=list)
    summary: str


# ============================================================
# Conversion helpers
# ============================================================

def to_review_event_record(
    record: CanonicalEventRecord | IndexedEventRecord | ReviewEventRecord,
) -> ReviewEventRecord:
    """
    Normalize any record-like object into the reviewer-facing shape.
    """
    return ReviewEventRecord(
        when=record.when,
        chapters=list(record.chapters),
        summary=record.summary,
    )


def build_event_review_decision(
    delta: CanonicalEventDelta,
    existing_events_by_uuid: Mapping[str, IndexedEventRecord],
) -> EventReviewDecision:
    """
    Convert one canonical delta into one reviewer-facing decision,
    enriching updates/deletes with the current indexed record from events.md.
    """
    action = delta.action

    if action == EventDeltaAction.CREATE:
        if delta.proposed_record is None:
            raise ValueError("CREATE delta must include proposed_record.")

        return EventReviewDecision(
            action=action,
            target_event_uuid=None,
            current_record=None,
            proposed_record=to_review_event_record(delta.proposed_record),
            identity_rationale=delta.reason,
            evidence_from_diff=list(delta.evidence_from_diff),
        )

    if action in (EventDeltaAction.UPDATE, EventDeltaAction.DELETE):
        if not delta.existing_event_uuid:
            raise ValueError(
                f"{action.value.upper()} delta must include existing_event_uuid."
            )

        current = existing_events_by_uuid.get(delta.existing_event_uuid)
        if current is None:
            raise KeyError(
                f"Delta references unknown existing event UUID: "
                f"{delta.existing_event_uuid}"
            )

        return EventReviewDecision(
            action=action,
            target_event_uuid=delta.existing_event_uuid,
            current_record=to_review_event_record(current),
            proposed_record=(
                to_review_event_record(delta.proposed_record)
                if delta.proposed_record is not None
                else None
            ),
            identity_rationale=delta.reason,
            evidence_from_diff=list(delta.evidence_from_diff),
        )

    raise ValueError(f"Unsupported delta action: {action!r}")


def build_approval_message(review_decisions: list[EventReviewDecision]) -> str:
    """
    Produce a short deterministic reviewer-facing approval summary.
    """
    if not review_decisions:
        return "No changes will be applied to events.md."

    counts = Counter(decision.action for decision in review_decisions)

    parts: list[str] = []

    create_count = counts.get(EventDeltaAction.CREATE, 0)
    update_count = counts.get(EventDeltaAction.UPDATE, 0)
    delete_count = counts.get(EventDeltaAction.DELETE, 0)

    if create_count:
        parts.append(f"{create_count} event(s) created")
    if update_count:
        parts.append(f"{update_count} event(s) updated")
    if delete_count:
        parts.append(f"{delete_count} event(s) deleted")

    joined = ", ".join(parts)
    return f"Approving will apply the following changes to events.md: {joined}."


# ============================================================
# Main adapter
# ============================================================

def build_events_review_proposal(
    canonical_proposal: EventsCanonDeltaProposal,
    existing_events_by_uuid: Mapping[str, IndexedEventRecord],
) -> EventsReviewProposal:
    """
    Deterministically convert the canonical LLM output into the
    reviewer-facing proposal.

    Rules:
    - changed is derived from whether canonical deltas exist
    - updates/deletes must resolve to existing indexed UUIDs
    - approval_message is generated in Python, not by the model
    """
    review_decisions = [
        build_event_review_decision(delta, existing_events_by_uuid)
        for delta in canonical_proposal.deltas
    ]

    changed = bool(review_decisions)

    return EventsReviewProposal(
        changed=changed,
        scan_summary=canonical_proposal.scan_summary,
        review_decisions=review_decisions,
        approval_message=build_approval_message(review_decisions),
    )

In [31]:
EVENTS_AGENT_SYSTEM_PROMPT = """
You are the Events Index Agent.

Your job is to maintain the canonical events index for the story world.

You must read:
1. the current contents of events.md
2. the incoming manuscript git diff

Then decide whether the diff implies any meaningful change to the canonical events index.

Core role:
You are not a creative writer.
You are not a story critic.
You are not a prose summarizer.
You are not a theorist inventing hidden meaning.

You are a careful canon librarian for events.

Your task is to perform canonical event recognition:
- identify meaningful world-relevant events in the diff
- decide whether each one is a new event or an update to an existing indexed event
- normalize each decision into a clean index-level event record
- stay grounded in the manuscript and the current events index

Definition of an event:
An event is a bounded, world-relevant unit of change.
It is something that happens, is discovered, is decided, or is revealed in a way that meaningfully changes the story world, a character's knowledge, a relationship, a goal, a risk, or the consequences that follow.
It is not merely a scene fragment, sentence-level motion, atmospheric detail, or isolated line of prose.
An event should be stable enough to remain the same event even if later chapters refine its timing, participants, cause, or full meaning.

What you must think about:
1. Did the diff introduce or clarify a meaningful event at the world-model level?
2. If yes, is it genuinely new, or is it the same event already present in events.md but now clearer?
3. What is the clean canonical record that should exist in events.md after review?
4. What exact evidence in the diff supports that decision?

Decision discipline:
- Prefer update over create when the diff is clearly expanding, clarifying, or sharpening an already indexed event.
- Do not create duplicate events just because the prose is richer, longer, more emotional, or more specific.
- Create a new event only when the diff supports a distinct bounded happening that is not already represented in the index.
- Be conservative and deliberate about event identity.

Index-level abstraction:
events.md is a compact canonical index, not a scene transcript.
Your summaries must be:
- short
- neutral
- world-level
- focused on what happened or became known

Do not include:
- literary commentary
- interpretation beyond what the manuscript supports
- emotional description unless it is part of the event itself
- unnecessary scene detail
- speculative meaning

Chronology rules:
The 'when' field is the best currently supported placement of the event in story time.
It may be:
- exact
- approximate
- relative
- inferred
- unknown

Do not force false precision.
Do not invent dates, timestamps, or chronology not grounded in the diff or current index.
If timing remains unclear, keep it open.
If the diff clarifies the timing of an existing event, prefer updating that event.

Chapters field:
The 'chapters' field is an internal grounding reference to where the event is evidenced, introduced, developed, or clarified in the manuscript.
It is not the meaning of the event.
Multiple chapter references are allowed when appropriate.

Summary field:
The 'summary' field must be a short canonical description of the event itself.
It should state the core happening or revelation in clean world-model language.
It should not read like prose from the manuscript.

Grounding rules:
- Stay faithful to the diff and the current events.md.
- Do not invent hidden events.
- Do not fill gaps with speculation.
- Do not infer more than the text can reasonably support.
- Every decision must be justifiable from the manuscript evidence.

Output rules:
- Return only the structured proposal in the required schema.
- If the diff does not imply any meaningful event-index change, set changed=false.
- If changed=true, each proposed decision must be explicit about whether it is a create or an update.
- For updates, make the existing match obvious and justified.
- Use compact evidence snippets from the diff to support each decision.

You are optimizing for:
- canonical consistency
- deduplication
- honest chronology
- compactness
- maintainability
- grounded judgment
"""

In [58]:
EVENTS_AGENT_USER_PROMPT_TEMPLATE = """
Current events.md:

<events_md>
{current_events_md}
</events_md>

Incoming manuscript diff:

<diff>
{diff_text}
</diff>

Task:
Review the diff against the current events index.

Decide whether the diff implies any meaningful change to the canonical events index.

If no meaningful event-index change is needed:
- return changed=false
- return no decisions

If a meaningful change is needed:
- identify each relevant event-level decision
- decide whether each decision is a create or an update
- for updates, match the event to the existing UUID in events.md
- propose the normalized record that should exist after approval
- include compact evidence from the diff for each decision

Return only the structured output in the required schema.
"""

In [44]:
HISTORY="""Attempt {attempt_number}

Previous proposal:
{serialize_structured_output(attempt.proposal)}

Reviewer feedback:
{attempt.reviewer_feedback}
"""

In [ ]:
class EventsFileParseError(ValueError):
    pass

